<a href="https://colab.research.google.com/github/Bodya-collab/LSTM-de-novo-Drug-Design-with-Reinforcement-Learning/blob/main/LSTM_cor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

# Connecting google drive
drive.mount('/content/drive')
work_dir = '/content/drive/MyDrive/Colab Notebooks/LSTM'

# 3. making a cd
os.chdir(work_dir)
print("-" * 40)
print(f"Working space: {os.getcwd()}")

##Basis for LSTM


In [ ]:
!pip install rdkit
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Input

vocab_size = 40
embedding_dim = 64
rnn_units = 128

# Architecture
lstm_generator = Sequential([
    Input(shape=(None,)),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    LSTM(rnn_units, return_sequences=True),
    Dense(vocab_size)
])

# Compilating
lstm_generator.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)
print("Stable model is compliated!")

#Downloading DF and Filtering + cleaning


In [ ]:
import pandas as pd

print("1. Downloading DF from AWS (DeepChem)...")
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/HIV.csv"
df_raw = pd.read_csv(url)

print(f"Downloaded molecules: {len(df_raw)}")

#Cleaning and preparing
allowed_chars = set('CON=()c12#SFCl[]')

def is_valid_for_my_model(smiles):
    # For too short and too long in order to prevent model abuse
    if len(smiles) < 15 or len(smiles) > 60:
        return False

    # 2. Veryfing Br,P,I
    for char in smiles:
        if char not in allowed_chars:
            return False

    return True

print("Filtering Dataset")

# Deep cleaning
df_clean = df_raw[df_raw['smiles'].apply(is_valid_for_my_model)]

# 20000 good molecules
sample_size = min(20000, len(df_clean))
df_final = df_clean.sample(n=sample_size, random_state=42).reset_index(drop=True)

# Saving on disk
df_final[['smiles']].to_csv('zinc_foundation_dataset.csv', index=False)

print("-" * 40)
print(f"Created CSV: zinc_foundation_dataset.csv")
print(f"Amount of molecules {len(df_final)}")
print("Examples:")
for s in df_final['smiles'].head(5):
    print(f" - {s}")

##Tokenization and pre-training

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

print("1. Читаем фундамент...")
df = pd.read_csv('zinc_foundation_dataset.csv')
smiles_list = df['smiles'].tolist()

# Vocabulary
vocab = ['C', 'O', 'N', '=', '(', ')', 'c', '1', '2', '#', 'S', 'F', 'Cl', '[', ']']
char2idx = {u: i for i, u in enumerate(vocab)}
vocab_size = len(vocab)
max_length = 60 # as previously set same lenght

print("Tokenization...")
X = []
Y = []
for smiles in smiles_list:
    seq = [char2idx.get(c, 0) for c in smiles]

    # Padding (adding 0 to make same lenght)
    if len(seq) < max_length + 1:
        seq += [0] * (max_length + 1 - len(seq))
    else:
        seq = seq[:max_length + 1]

    X.append(seq[:-1])
    Y.append(seq[1:])

X_train = np.array(X)
Y_train = np.array(Y)

print("Creating a model (LSTM)...")
def build_clean_model(vocab_size, embedding_dim=128, rnn_units=256):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
        tf.keras.layers.LSTM(rnn_units, return_sequences=True),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

foundation_model = build_clean_model(vocab_size)

#Predicting the next symbol (optimization)
foundation_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("Pre-training...")
# Deep lern
history = foundation_model.fit(
    X_train,
    Y_train,
    epochs=30,
    batch_size=256,      # 256 mol per s
    validation_split=0.1 # 10% of validation
)

print("\nDone")

# Saving
foundation_model.save('foundation_lstm_zinc.keras')
print("Saved as: 'foundation_lstm_zinc.keras'")

##XGBoost and ESOL

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from rdkit import Chem
from rdkit.Chem import Descriptors

print("1. Downloading dataset ESOL (solubility)...")
# 1100 molecules with measured LogS)
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv"
df_esol = pd.read_csv(url)

print("2. Preparing descriptorsc...")
# Transforming smiles in numbers for XGBoost
def get_advanced_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol)
    ])

# X - fiches, Y - solubility)
X_data = []
y_data = []

for index, row in df_esol.iterrows():
    smiles = row['smiles']
    real_logs = row['measured log solubility in mols per litre']

    features = get_advanced_features(smiles)
    if features is not None:
        X_data.append(features)
        y_data.append(real_logs)

X = np.array(X_data)
y = np.array(y_data)

print(f"Extracted {len(X)} molecules. Studying...")

# 3. ML
# Classic hyperparameters.
model_xgb = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

model_xgb.fit(X, y)

# 4. Saving
joblib.dump(model_xgb, 'xgboost_logs_model.pkl')

print("-" * 40)
print("Saved as 'xgboost_logs_model.pkl'.")

## Reward function applying

In [ ]:
import joblib
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED
import tensorflow as tf
import numpy as np

print("1. Uploading XGBoost...")
model_tuned = joblib.load('xgboost_logs_model.pkl')

def calculate_reward_fast(smiles, precalculated_logs):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0.0, {"QED": 0, "LogS": 0, "Valid": False}

    mw = Descriptors.MolWt(mol)
    smiles_len = len(smiles)
    ring_info = mol.GetRingInfo().NumRings()

    penalty = 1.0

    # Penalty if MW is too low or high
    if mw < 200 or smiles_len < 15: penalty *= 0.1
    if mw > 500 or smiles_len > 45: penalty *= 0.1

    c_count = smiles.count('C') + smiles.count('c')
    if smiles_len > 0 and (c_count / smiles_len) > 0.75: penalty *= 0.1

    # Heteroatomic bonuses (O, N)
    complexity_bonus = 0.2 if (ring_info > 0 and ('O' in smiles or 'N' in smiles or 'n' in smiles)) else 0.0

    qed_score = QED.qed(mol)

    # FIXED: Use the precalculated logS directly
    logs_pred = precalculated_logs

    # Bonus for good solubility
    logs_bonus = 0.5 if -5.0 <= logs_pred <= -1.0 else 0.0

    base_reward = (qed_score * 0.7) + (logs_bonus * 0.3)
    final_reward = (base_reward + complexity_bonus) * penalty

    details = {
        "QED": round(qed_score, 3),
        "LogS": round(logs_pred, 2),
        "Valid": True
    }
    return round(min(final_reward, 1.0), 3), details

##Basis for generation and tokenization for RL

In [ ]:
import numpy as np
import tensorflow as tf
from rdkit import Chem

# 1. Apploading vocabulary
vocab = ['C', 'O', 'N', '=', '(', ')', 'c', '1', '2', '#', 'S', 'F', 'Cl', '[', ']']
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)
max_length = 60

def prepare_data_for_lstm(smiles_list, max_length=60):
    X_list, Y_list = [], []
    for smiles in smiles_list:
        seq = [char2idx.get(c, 0) for c in smiles]
        if len(seq) < max_length + 1:
            seq += [0] * (max_length + 1 - len(seq))
        else:
            seq = seq[:max_length + 1]
        X_list.append(seq[:-1])
        Y_list.append(seq[1:])
    return np.array(X_list), np.array(Y_list)

def generate_smiles_batch(model, num_samples, max_length=60, temperature=1.0):
    generated_matrix = np.zeros((num_samples, max_length), dtype=np.int32)
    generated_matrix[:, 0] = char2idx['C']
    for i in range(max_length - 1):
        predictions = model(generated_matrix, training=False)
        next_char_logits = predictions[:, i, :] / temperature
        next_chars = tf.random.categorical(next_char_logits, num_samples=1).numpy()[:, 0]
        generated_matrix[:, i + 1] = next_chars
    batch_smiles = []
    for row in generated_matrix:
        text = "".join([idx2char[c] for c in row])
        if 'C' in text: text = text.rstrip('C') if text.endswith('C') else text
        batch_smiles.append(text)
    return batch_smiles

EPOCHS = 10
BATCH_SIZE = 250
current_generator = foundation_model

print("Predicting via RL")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS} for data...")
    generated_batch = generate_smiles_batch(current_generator, BATCH_SIZE, temperature=1.0)
    valid_mols, features_list = [], []

    for sm in generated_batch:
        mol = Chem.MolFromSmiles(sm)
        if mol is not None:
            f = get_advanced_features(sm)
            if f is not None:
                valid_mols.append(sm)
                features_list.append(f)

    scored_molecules = []
    if len(features_list) > 0:
        all_logs = model_tuned.predict(np.array(features_list))
        for i, sm in enumerate(valid_mols):
            reward_tuple = calculate_reward_fast(sm, all_logs[i])
            scored_molecules.append({"smiles": sm, "reward": reward_tuple[0]})

    scored_molecules.sort(key=lambda x: x["reward"], reverse=True)
    real_elite = [m for m in scored_molecules if m["reward"] >= 0.1]
    print(f"Created: {len(valid_mols)}, picked: {len(real_elite)}")

    if len(real_elite) > 0:
        X_elite, Y_elite = prepare_data_for_lstm([m['smiles'] for m in real_elite])
        current_generator.fit(X_elite, Y_elite, epochs=1, batch_size=32, verbose=0)
    else:
        X_elite, Y_elite = prepare_data_for_lstm(golden_smiles)
        current_generator.fit(X_elite, Y_elite, epochs=1, batch_size=4, verbose=0)

##RL-cycle

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, QED

# Redefine the function to correctly use the precalculated log solubility
def calculate_reward_fast(smiles, logs_pred):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0.0

    mw = Descriptors.MolWt(mol)
    smiles_len = len(smiles)
    ring_info = mol.GetRingInfo().NumRings()

    penalty = 1.0

    if mw < 200 or smiles_len < 15: penalty *= 0.1
    if mw > 500 or smiles_len > 45: penalty *= 0.1

    # Penalty for spamming C
    c_count = smiles.count('C') + smiles.count('c')
    if smiles_len > 0 and (c_count / smiles_len) > 0.75: penalty *= 0.1

    # Due to high abuse of sulfur new clause (only 1 S in compound)
    s_count = smiles.count('S') + smiles.count('s')
    if s_count > 1: penalty *= 0.01

    # Previously a lot of S=S compunds (banned)
    if 'SS' in smiles or 'ss' in smiles: penalty *= 0.01

    complexity_bonus = 0.2 if (ring_info > 0 and ('O' in smiles or 'N' in smiles or 'n' in smiles)) else 0.0

    qed_score = QED.qed(mol)
    logs_bonus = 0.5 if -5.0 <= logs_pred <= -1.0 else 0.0

    base_reward = (qed_score * 0.7) + (logs_bonus * 0.3)
    final_reward = (base_reward + complexity_bonus) * penalty

    return round(min(final_reward, 1.0), 3)

EPOCHS = 10
BATCH_SIZE = 250

golden_smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",
    "CC1=CC=C(C=C1)NC(=O)C",
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "c1ccccc1"
]

print("--- STARTING RL CYCLE WITH RESULTS SAVING ---")

current_generator = foundation_model
#elite structures are sent to the filtered set
all_generated_elites = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS} started...")

    generated_batch = generate_smiles_batch(current_generator, BATCH_SIZE, temperature=1.0)

    valid_mols = []
    features_list = []

    for sm in generated_batch:
        mol = Chem.MolFromSmiles(sm)
        if mol is not None:
          f = get_advanced_features(sm)
          if f is not None:
                valid_mols.append(sm)
                features_list.append(f)

    scored_molecules = []
    if len(features_list) > 0:
        all_logs = model_tuned.predict(np.array(features_list))

        for i, sm in enumerate(valid_mols):
            # FIXED: reward is now a single float
            reward = calculate_reward_fast(sm, all_logs[i])
            scored_molecules.append({"smiles": sm, "reward": reward})

    # FIXED: Access reward directly, not as reward[0]
    scored_molecules.sort(key=lambda x: x["reward"], reverse=True)

    if len(scored_molecules) > 0:
        best_raw = scored_molecules[0]
        print(f"[Debug] Top of batch: {best_raw['smiles']} (Reward: {best_raw['reward']})")

    real_elite = [m for m in scored_molecules if m["reward"] >= 0.1]

    print(f"From {BATCH_SIZE} molecules, {len(real_elite)} passed the filter")

    for item in real_elite:
        all_generated_elites.append({
            "epoch": epoch + 1,
            "smiles": item["smiles"],
            "ai_score": item["reward"]
        })

    elite_smiles_list = [item["smiles"] for item in real_elite]

    if len(real_elite) == 0:
        print("No elite molecules found. Using golden molecules for correction.")
        elite_smiles_list.extend(golden_smiles)

    X_elite, Y_elite = prepare_data_for_lstm(elite_smiles_list)
    current_generator.fit(X_elite, Y_elite, epochs=1, batch_size=32, verbose=0)

print("\n" + "="*40)
if len(all_generated_elites) > 0:
    df_results = pd.DataFrame(all_generated_elites)
    df_results = df_results.drop_duplicates(subset=['smiles'])
    df_results = df_results.sort_values(by='ai_score', ascending=False)
    df_results.to_csv('my_ai_molecules.csv', index=False)
    print(f"SUCCESS! Cycle completed.")
    print(f"Unique elite structures collected: {len(df_results)}")
    print(f"Best generated reward: {df_results['ai_score'].iloc[0]}")
else:
    print("Cycle completed, but no elite molecules were collected.")

##New code to take a look on a ML progress

In [ ]:
!pip install rdkit

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, QED
import tensorflow as tf
import os

# Check if model is in memory, if not - load from disk
if 'foundation_model' not in locals():
    print("Loading foundation_model from disk...")
    if os.path.exists('foundation_lstm_zinc.keras'):
        foundation_model = tf.keras.models.load_model('foundation_lstm_zinc.keras')
    else:
        print("Error: 'foundation_lstm_zinc.keras' not found. Please run the pre-training cell first.")

# Redefine the function to correctly use the precalculated log solubility
def calculate_reward_fast(smiles, logs_pred):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0.0

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    rot_bonds = Descriptors.NumRotatableBonds(mol)
    smiles_len = len(smiles)
    ring_info = mol.GetRingInfo().NumRings()

    penalty = 1.0

    if mw < 200 or smiles_len < 15: penalty *= 0.1
    if mw > 500 or smiles_len > 45: penalty *= 0.1

    if logp > 4.0 or logp < 0.0: penalty *= 0.5
    if rot_bonds > 6: penalty *= 0.2

    s_count = smiles.count('S') + smiles.count('s')
    if s_count > 0: penalty *= 0.1

    o_count = smiles.count('O') + smiles.count('o')
    if o_count > 4: penalty *= 0.5

    c_count = smiles.count('C') + smiles.count('c')
    if smiles_len > 0 and (c_count / smiles_len) > 0.70: penalty *= 0.1

    complexity_bonus = 0.2 if (ring_info > 0 and ('N' in smiles or 'n' in smiles)) else 0.0
    qed_score = QED.qed(mol)
    logs_bonus = 0.5 if -5.0 <= logs_pred <= -1.0 else 0.0

    base_reward = (qed_score * 0.7) + (logs_bonus * 0.3)
    final_reward = (base_reward + complexity_bonus) * penalty
    return round(min(final_reward, 1.0), 3)

EPOCHS = 10
BATCH_SIZE = 1000
golden_smiles = ["CC(=O)OC1=CC=CC=C1C(=O)O", "CC1=CC=C(C=C1)NC(=O)C", "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "c1ccccc1"]

print("--- STARTING RL CYCLE WITH RESULTS SAVING ---")
current_generator = foundation_model
all_generated_elites = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS} started...")
    generated_batch = generate_smiles_batch(current_generator, BATCH_SIZE, temperature=1.0)
    valid_mols, features_list = [], []

    for sm in generated_batch:
        mol = Chem.MolFromSmiles(sm)
        if mol is not None:
          f = get_advanced_features(sm)
          if f is not None:
                valid_mols.append(sm)
                features_list.append(f)

    scored_molecules = []
    if len(features_list) > 0:
        all_logs = model_tuned.predict(np.array(features_list), verbose=0)
        for i, sm in enumerate(valid_mols):
            reward = calculate_reward_fast(sm, all_logs[i])
            scored_molecules.append({"smiles": sm, "reward": reward})

    scored_molecules.sort(key=lambda x: x["reward"], reverse=True)
    if len(scored_molecules) > 0:
        best_raw = scored_molecules[0]
        print(f"[Debug] Top of batch: {best_raw['smiles']} (Reward: {best_raw['reward']})")

    real_elite = [m for m in scored_molecules if m["reward"] >= 0.1]
    print(f"From {BATCH_SIZE} molecules, {len(real_elite)} passed the filter")

    for item in real_elite:
        all_generated_elites.append({"epoch": epoch + 1, "smiles": item["smiles"], "ai_score": item["reward"]})

    elite_smiles_list = [item["smiles"] for item in real_elite]
    if len(real_elite) == 0: elite_smiles_list.extend(golden_smiles)

    X_elite, Y_elite = prepare_data_for_lstm(elite_smiles_list)
    current_generator.fit(X_elite, Y_elite, epochs=1, batch_size=32, verbose=0)

print("\nSUCCESS! Cycle completed.")
if len(all_generated_elites) > 0:
    df_results = pd.DataFrame(all_generated_elites).drop_duplicates(subset=['smiles']).sort_values(by='ai_score', ascending=False)
    df_results.to_csv('my_ai_molecules.csv', index=False)
    print(f"Unique elite structures collected: {len(df_results)}")

##Result display

In [ ]:
from rdkit.Chem import Draw
import pandas as pd

# Uploading results
try:
    df_res = pd.read_csv('my_ai_molecules.csv')
    print(f"Unique molecules: {len(df_res)}")

    # head top 12
    top_mols_data = df_res.head(12)
    mols = [Chem.MolFromSmiles(s) for s in top_mols_data['smiles']]
    labels = [f"Score: {r:.3f}" for r in top_mols_data['ai_score']]

    print("Winner`s visualization")
    img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(300, 200), legends=labels)
    display(img)
except FileNotFoundError:
    print("Nothing to show")

#QED score

In [ ]:
from rdkit.Chem import QED
import pandas as pd

print("Counting QED...")
df_res['QED'] = df_res['smiles'].apply(lambda sm: QED.qed(Chem.MolFromSmiles(sm)) if Chem.MolFromSmiles(sm) else 0)

# Saving with new table
df_res.to_csv('my_ai_molecules_full_QED.csv', index=False)
print("Saved as: my_ai_molecules_full.csv")
df_res = pd.read_csv('my_ai_molecules_full_QED.csv')
df_res.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    # Load the results generated in the previous steps
    df_res = pd.read_csv("my_ai_molecules.csv")

    # Select the top 12 candidates based on the ai_score
    top_12_df = df_res.head(12)

    # Save to a new CSV file
    top_12_filename = "top_12_molecules.csv"
    top_12_df.to_csv(top_12_filename, index=False)

    # Plot histogram of ai_score
    plt.figure(figsize=(10, 6))
    sns.histplot(df_res["ai_score"], bins=30, kde=True, color="teal")
    plt.title("Distribution of AI Scores (Reward)")
    plt.xlabel("AI Score")
    plt.ylabel("Count")
    plt.show()

except FileNotFoundError:
    print("Error: 'my_ai_molecules.csv' not found.")

##Saving top 12 molecules with highest Ai_score

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd

try:
    # Load the top 12 molecules
    df_top = pd.read_csv('top_12_molecules.csv')

    sdf_filename = 'top_12_molecules_maestro.sdf'
    writer = Chem.SDWriter(sdf_filename)

    count = 0
    for index, row in df_top.iterrows():
        mol = Chem.MolFromSmiles(row['smiles'])
        if mol:
            # Generate 2D coordinates for clear display in Maestro
            AllChem.Compute2DCoords(mol)

            # Add metadata fields
            mol.SetProp("_Name", f"AI_Mol_{index+1}")
            mol.SetProp("AI_Score", str(row['ai_score']))
            mol.SetProp("Generation_Epoch", str(row['epoch']))

            writer.write(mol)
            count += 1

    writer.close()
    print(f" {count} SDF file: {sdf_filename}")
except FileNotFoundError:
    print("Error, no CSV file detected")

##Ai_score cor with Amounts of rings

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem

try:
    # Load the results
    df_res = pd.read_csv('my_ai_molecules.csv')

    # Function to calculate ring count
    def get_ring_count(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return mol.GetRingInfo().NumRings()
        return 0

    print('Calculating ring counts for all molecules...')
    df_res['ring_count'] = df_res['smiles'].apply(get_ring_count)

    # Calculate Pearson Correlation
    correlation = df_res['ai_score'].corr(df_res['ring_count'])

    # Visualization
    plt.figure(figsize=(10, 6))
    sns.regplot(x='ring_count', y='ai_score', data=df_res, x_jitter=0.2, scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
    plt.title(f'Correlation Analysis (Pearson: {correlation:.4f})')
    plt.xlabel('Number of Rings')
    plt.ylabel('AI Score (Reward)')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    print(f'Correlation Coefficient: {correlation:.4f}')

    # Display summary table
    display(df_res.groupby('ring_count')['ai_score'].agg(['mean', 'count']))

except FileNotFoundError:
    print('Error: my_ai_molecules.csv not found. Please run the RL cycle first.')

##Ai_score cor with heteroatomics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem

try:
    df_res = pd.read_csv('my_ai_molecules.csv')

    def get_heteroatom_count(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            # Heteroatoms are non-carbon, non-hydrogen atoms
            return sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() not in ['C', 'H'])
        return 0

    print('Calculating heteroatom counts...')
    df_res['heteroatom_count'] = df_res['smiles'].apply(get_heteroatom_count)

    correlation = df_res['ai_score'].corr(df_res['heteroatom_count'])

    plt.figure(figsize=(10, 6))
    sns.regplot(x='heteroatom_count', y='ai_score', data=df_res, x_jitter=0.2, scatter_kws={'alpha':0.3}, line_kws={'color':'black'})
    plt.title(f'Heteroatom Count vs AI Score (Pearson: {correlation:.4f})')
    plt.xlabel('Number of Heteroatoms (O, N, S, etc.)')
    plt.ylabel('AI Score')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

    print(f'Correlation Coefficient: {correlation:.4f}')
    display(df_res.groupby('heteroatom_count')['ai_score'].agg(['mean', 'count']))

except FileNotFoundError:
    print('Error: my_ai_molecules.csv not found.')

##Lipinski Rule MW vs LogP

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit import DataStructs
import numpy as np

print("Counting descriptors for analysis...")

weights, logps, lengths = [], [], []
valid_mols = []

for sm in df_res['smiles']:
    mol = Chem.MolFromSmiles(sm)
    if mol:
        weights.append(Descriptors.MolWt(mol))
        logps.append(Descriptors.MolLogP(mol))
        lengths.append(len(sm))
        valid_mols.append(mol)

df_res['MolWt'] = weights
df_res['LogP'] = logps
df_res['SMILES_Length'] = lengths

# 1. MW vs LogP
plt.figure(figsize=(10, 6))
sns.scatterplot(x=df_res['MolWt'], y=df_res['LogP'], hue=df_res['ai_score'], palette='viridis', alpha=0.7)
plt.axvline(x=500, color='r', linestyle='--', label='Max Weight (500)')
plt.axhline(y=5, color='r', linestyle=':', label='Max LogP (5)')
plt.title('Chemical Space (Lipinski rule)')
plt.xlabel('Molecular weight')
plt.ylabel('LogP')
plt.legend()
plt.show()

##Smiles_length vs AI_score (veryfing_cheating_behave)

In [ ]:
# 2. Graph: Smile`s lenght vs AI score
plt.figure(figsize=(10, 6))
sns.regplot(x=df_res['SMILES_Length'], y=df_res['ai_score'], scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title('Correlation:Smile`s lenght and AI score')
plt.xlabel('amount of symbols')
plt.ylabel('AI Score')
plt.show()

##Tanimoto Similarity

In [ ]:
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
import numpy as np

# Ensure we have a sample of molecules to check similarity
# We use valid_mols created in the Lipinski cell
sample_size = min(5000, len(valid_mols))
sample_mols = np.random.choice(valid_mols, sample_size, replace=False)

print(f"Calculating Tanimoto Similarity for {sample_size} molecules...")

# Create Morgan Fingerprint generator
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [mfpgen.GetFingerprint(m) for m in sample_mols]

similarities = []
for i in range(len(fps)):
    for j in range(i + 1, len(fps)):
        similarities.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))

avg_similarity = np.mean(similarities)
print(f"Average similarity: {avg_similarity:.3f}")

if avg_similarity < 0.4:
    print("-> Good: high structural diversity (dissimilarity).")
elif avg_similarity < 0.7:
    print("-> Warning: some clusters of similar molecules detected.")
else:
    print("-> Alert: Mode Collapse. The model is generating very similar structures.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

try:
    # Selecting only numerical columns for correlation analysis
    # We include properties we've calculated in previous steps
    numerical_cols = ['ai_score', 'MolWt', 'LogP', 'SMILES_Length', 'ring_count', 'heteroatom_count']

    # Ensure the columns exist in df_res
    available_cols = [col for col in numerical_cols if col in df_res.columns]
    corr_matrix = df_res[available_cols].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Heatmap of Molecular Features and AI Score')
    plt.show()

    # Saving the final tuned model as requested in the task context
    current_generator.save('tuned_lstm_zinc.keras')
    print("Model weights saved as 'tuned_lstm_zinc.keras'")

except Exception as e:
    print(f"An error occurred: {e}")

#Importing CSV for R studio

In [ ]:
df_res.to_csv('my_ai_molecules_full.csv', index=False)